# RG3 가스: 정상 운전 조건 대비 센서 관계 잔차

평가 대상 RG3 RH 620개(가스19/정상601)만 사용. CN7 학습/평가 미사용. 속도·RPM·배럴2/5·계량위치·충전시간 6개로 정상의 최대사출압력·전환압력·쿠션·계량시간 4개 기대값을 회귀. 실제값-기대값의 부호/절댓값 8개 파생 특징. 정상에서만 관계 모델/결측 처리/잔차 스케일 학습. 가스 원인 확정 또는 물리 법칙 모델은 아님.

잔차8, std4 비교, 두 전문가 최대 점수 앙상블. RF/ExtraTrees 2시드 평균. 외부 관측구간은 전처리·관계회귀·분류·임계값 선택 모두 제외. 7개 관측구간 중첩 검증이며 실제 Run 확정 구분/새 독립 검증 아님. 역사적 17개 평가와 모집단 및 검증 기준 다름. 과거10개 표준편차는 이전 센서 관측만 사용하지만 완전히 배포용 롤링 검증과 동일하지 않음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][inputs+targets+stdcols])
 tr=im.transform(d.iloc[ref][inputs+targets+stdcols]);te=im.transform(d.iloc[q][inputs+targets+stdcols])
 nt=tr[y[ref]==0]
 relation=ExtraTreesRegressor(n_estimators=80,min_samples_leaf=12,max_depth=5,random_state=42,n_jobs=2).fit(nt[:,:6],nt[:,6:10])
 scale=np.maximum(np.quantile(nt[:,6:10],.75,axis=0)-np.quantile(nt[:,6:10],.25,axis=0),1e-6)
 rt=(tr[:,6:10]-relation.predict(tr[:,:6]))/scale
 rq=(te[:,6:10]-relation.predict(te[:,:6]))/scale
 features=[(tr[:,10:],te[:,10:]),(np.c_[rt,np.abs(rt)],np.c_[rq,np.abs(rq)])]
 scores=[]
 for ft,fq in features:
  vals=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    if len(np.unique(y[ref]))<2:vals.append(np.zeros(len(q)));continue
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],np.maximum(*scores)])
 audit.append(dict(train=len(ref),query=len(q),normal_relation_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','conditional_residual8','std4_residual_max_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_conditional_residual_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                      model  budget  TP  FP  FN  TN   recall       F1       AP
      conditional_residual8    0.02   0  13  19 588 0.000000 0.000000 0.023842
      conditional_residual8    0.05   0  35  19 566 0.000000 0.000000 0.023842
      conditional_residual8    0.10   1  71  18 530 0.052632 0.021978 0.023842
inner_recall_first_selected    0.00   3  63  16 538 0.157895 0.070588      NaN
             std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
             std4_reference    0.05   3  33  16 568 0.157895 0.109091 0.061578
             std4_reference    0.10   3  67  16 534 0.157895 0.067416 0.061578
 std4_residual_max_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.053357
 std4_residual_max_ensemble    0.05   3  33  16 568 0.157895 0.109091 0.053357
 std4_residual_max_ensemble    0.10   3  62  16 539 0.157895 0.071429 0.053357
